# Tipos e extração de dados

Transformar dados textuais, JSON e textos livres em tabelas com tipos adequados.

Execute as células em ordem. Este notebook foi revisado a partir de uma atividade acadêmica para apresentação em portfólio.

## Conversão de dados de produtos

In [1]:
# Entrada de dados

dados_brutos = [
    {
        "produto": "Notebook Gamer",
        "avaliacao": "Excelente",
        "desconto": "15%",
        "estoque": "12 unidades",
        "largura_cm": "35.5",
    },
    {
        "produto": "Mouse Sem Fio",
        "avaliacao": "Bom",
        "desconto": "0%",
        "estoque": "Out-of-stock",
        "largura_cm": "12.0",
    },
    {
        "produto": "Teclado Mecânico",
        "avaliacao": "Regular",
        "desconto": "5%",
        "estoque": "3 unidades",
        "largura_cm": "44.2",
    },
    {
        "produto": "Monitor",
        "avaliacao": "Regular",
        "desconto": "12%",
        "estoque": "30 unidades",
        "largura_cm": "52.2",
    }
]


In [2]:
import pandas as pd

df = pd.DataFrame(dados_brutos) # cria data frame com os dados brutos
df["desconto"] = df["desconto"].str.replace("%","") # retira o simbolo de porcentagem em cada campo "desconto"
df["desconto"] = df["desconto"].astype(float) # transforma o número, agora sem o simbolo de porcentagem, para float
df["desconto"] = df["desconto"] / 100 #divide por cem para deixar no formato exigido

df["estoque"] = df["estoque"].str.replace("Out-of-stock", "0") # transforma out-of stock em 0
df["estoque"] = df["estoque"].str.replace(" unidades", "") # retira a palavra unidades dos demais
df["estoque"] = df["estoque"].astype(int) # transforma em inteiro

df["avaliacao"] = pd.Categorical(df["avaliacao"], categories = ["Ruim", "Regular", "Bom", "Excelente"], ordered=True)  # método Categorical transforma "avaliacao" em uma variável qualitativa ordinal, definindo a ordem Ruim < Regular < Bom < Excelente

df["largura_cm"] = pd.to_numeric(df["largura_cm"])
df # Exibe os dados transformados


,produto,avaliacao,desconto,estoque,largura_cm
0,Notebook Gamer,Excelente,0.15,12,35.5
1,Mouse Sem Fio,Bom,0.00,0,12.0
2,Teclado Mecânico,Regular,0.05,3,44.2
3,Monitor,Regular,0.12,30,52.2


In [3]:
df.dtypes # imprime os tipos de dados do data frame


produto            str
avaliacao     category
desconto       float64
estoque          int64
largura_cm     float64
dtype: object

## Normalização de JSON meteorológico

In [4]:
# Resposta da API
response_api = {
  "cidade": "São Paulo",
  "coordenadas": {"lat": -23.55, "lon": -46.63},
  "previsoes": [
    {"data": "2026-03-01", "temp_max": "28.4°C", "humidade": "75%", "condicao": "Chuvoso"},
    {"data": "2026-03-02", "temp_max": "31.0°C", "humidade": "60%", "condicao": "Ensolarado"}
  ]
}

result = pd.json_normalize(response_api, record_path="previsoes", meta=["cidade", ["coordenadas", "lat"], ["coordenadas", "lon"]]) # retira as informações aninhadas e transforma em uma estrutura tabular

result["data"] = pd.to_datetime(result["data"]) # transforma o campo data, antes string, para o tipo temporal adequado

result["temp_max"] = result["temp_max"].str.replace("°C", "") # retira o simbolo de temperatura, deixando apenas a parte numerica
result["temp_max"] = result["temp_max"].astype(float) # transforma para o tipo float

result["humidade"] = result["humidade"].str.replace("%", "") #retira o simbolo de porcentagem, deixando apenas a parte numerica
result["humidade"] = result["humidade"].astype(int) # transforma para o tipo int

display(result)
result.dtypes #imprime os tipos de variávies


,data,temp_max,humidade,condicao,cidade,coordenadas.lat,coordenadas.lon
0,2026-03-01,28.4,75,Chuvoso,São Paulo,-23.55,-46.63
1,2026-03-02,31.0,60,Ensolarado,São Paulo,-23.55,-46.63


data               datetime64[us]
temp_max                  float64
humidade                    int64
condicao                      str
cidade                        str
coordenadas.lat            object
coordenadas.lon            object
dtype: object

## Extração de logs com expressões regulares

In [5]:
import re

log_servico_web = """192.168.1.10 - - [01/Mar/2026:10:15:30 -0300] "GET /produtos/item1 HTTP/1.1" 200 4520
                      10.0.0.5 - - [01/Mar/2026:10:16:02 -0300] "POST /carrinho/add HTTP/1.1" 500 120
                      192.168.1.10 - - [01/Mar/2026:10:18:45 -0300] "GET /checkout HTTP/1.1" 200 8900"""

pattern = r'(?P<ip>\S+) - - \[(?P<timestamp>[^\]]+)\] "(?P<method>\S+) (?P<path>\S+) (?P<protocolo>\S+)" (?P<status>\S+) (?P<bytes>\S+)' #definição da pattern para extração dos dados

resultado = re.findall(pattern, log_servico_web) # extração aos moldes da pattern

df = pd.DataFrame(resultado, columns=["ip", "timestamp", "method", "path", "protocolo", "status", "bytes"]) # criação d0 dataframe com as colunas indicadas

df["ip"] = df["ip"].astype("string") #tratamento de ip para string

df["timestamp"] = pd.to_datetime(df["timestamp"], format="%d/%b/%Y:%H:%M:%S %z") # tratamento de timestamp para o formato exigido

df["bytes"] = df["bytes"].astype(int) # tratamento de bytes para inteiro

df["status"] = df["status"].astype(int)
df.dtypes # imprime os tipos


ip                              string
timestamp    datetime64[us, UTC-03:00]
method                             str
path                               str
protocolo                          str
status                           int64
bytes                            int64
dtype: object

In [6]:
df


,ip,timestamp,method,path,protocolo,status,bytes
0,192.168.1.10,2026-03-01 10:15:30-03:00,GET,/produtos/item1,HTTP/1.1,200,4520
1,10.0.0.5,2026-03-01 10:16:02-03:00,POST,/carrinho/add,HTTP/1.1,500,120
2,192.168.1.10,2026-03-01 10:18:45-03:00,GET,/checkout,HTTP/1.1,200,8900


## Extração de informações de atas

In [7]:
atas_reuniao = """
=== ATA DE REUNIÃO - 12/03/2026 ===
Estiveram presentes Carlos, Amanda e a equipe de Operações. Discutiu-se longamente a necessidade urgente de expansão do hub Sul devido ao aumento de 35% na demanda. Amanda apresentou três cotações de mercado e defendeu a frota própria.
#PROJETO: Expansão Logística
Após duas horas de debate sobre viabilidade financeira e sustentabilidade, o comitê definiu o caminho. [DECISÃO: Contratação imediata de nova frota de caminhões elétricos para o hub Sul]. O diretor financeiro confirmou que o processo está com [STATUS: Aprovado].
Para a execução da compra, ficou definido o investimento de [ORÇAMENTO: R$ 450.000,00]. A operação possui um [RISCO: Médio] por conta da dependência de postos de recarga e deve ser concluída no prazo de [PRAZO: 60 dias].

=== ATA DE REUNIÃO - 15-03-2026 ===
Reunião extraordinária convocada pelo time de Segurança da Informação. O foco principal foi revisar a viabilidade do projeto #PROJETO: Migração Cloud ainda no primeiro semestre.
O time técnico demonstrou preocupação com possíveis instabilidades durante a transição. Diante dos alertas, a diretoria tomou uma posição preventiva. [DECISÃO: Postergar a migração do banco de dados legado] até o terceiro trimestre. Desta forma, o alinhamento atual permanece [STATUS: Em Análise].
O valor previsto para cobrir a consultoria de apoio é de [ORÇAMENTO: R$ 120.000,00]. O projeto foi classificado com [RISCO: Alto] e terá uma janela estendida com [PRAZO: 90 dias] para a realização de novos testes de estresse.
        """


In [8]:
# Separa cada ata antes de extrair os campos para evitar cruzar registros.
registros = []
for ata in atas_reuniao.split("=== ATA DE REUNIÃO - ")[1:]:
    registro = {"DATA": re.search(r"\d{2}[/-]\d{2}[/-]\d{4}", ata).group()}
    registro["PROJETO"] = re.search(r"#PROJETO:\s*(.+?)(?: ainda\b|\n|$)", ata).group(1).strip()
    for campo in ["DECISÃO", "STATUS", "ORÇAMENTO", "RISCO", "PRAZO"]:
        registro[campo] = re.search(r"\[" + campo + r":\s*([^\]]+)\]", ata).group(1).strip()
    registros.append(registro)

df = pd.DataFrame(registros)
df["DATA"] = pd.to_datetime(df["DATA"].str.replace("-", "/", regex=False), format="%d/%m/%Y")
for coluna in ["PROJETO", "DECISÃO", "STATUS"]:
    df[coluna] = df[coluna].astype("string")
df["ORÇAMENTO"] = (df["ORÇAMENTO"].str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False).str.replace(",", ".", regex=False).str.strip().astype(float))
df["RISCO"] = pd.Categorical(df["RISCO"], categories=["Baixo", "Médio", "Alto"], ordered=True)
df["PRAZO"] = df["PRAZO"].str.extract(r"(\d+)", expand=False).astype(int)
display(df)
df.dtypes


,DATA,PROJETO,DECISÃO,STATUS,ORÇAMENTO,RISCO,PRAZO
0,2026-03-12,Expansão Logística,Contratação imediata de nova frota de caminhõe...,Aprovado,450000.0,Médio,60
1,2026-03-15,Migração Cloud,Postergar a migração do banco de dados legado,Em Análise,120000.0,Alto,90


DATA         datetime64[us]
PROJETO              string
DECISÃO              string
STATUS               string
ORÇAMENTO           float64
RISCO              category
PRAZO                 int64
dtype: object

## Escopo

Os dados são exemplos sintéticos embutidos no notebook. A extração das atas foi adaptada ao padrão desses textos, incluindo o delimitador “ainda” no segundo nome de projeto; não é um extrator universal de atas.